In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

✅ 로컬 Jupyter


In [2]:
# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


In [3]:
# 패키지 import

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")

✅ import 완료


In [4]:
import json
with open("personas.json", encoding="utf-8") as f:
    PERSONAS = json.load(f)

In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}")
])   # LCEL 파이프 연산자

# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    persona = PERSONAS[persona_name]

    system_message = persona["system"]
    temperature = persona["temperature"]

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=temperature, max_tokens=200)

    chain = prompt | llm

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})

    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

In [7]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /list, /help")

    elif cmd == "/list":
        print("🎭 사용 가능한 페르소나:")
        
        for name, info in PERSONAS.items():
            print(f"- {name}: {info['description']}")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

In [9]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

In [10]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 토큰 300개를 작성하는 것은 특정한 주제나 내용을 기반으로 해야 의미가 있습니다. 어떤 주제에 대해 300개의 토큰을 작성해 드릴까요? 예를 들어, 데이터 분석, 머신러닝, 특정 산업 동향 등 다양한 주제가 가능합니다. 원하는 주제를 말씀해 주시면 그에 맞춰 작성하겠습니다.

✅ 페르소나가 '디자이너'(으)로 변경되었습니다.
[디자이너] 물론입니다! 사용자 경험(UX) 디자인에 대한 내용을 시각적 비유를 통해 설명해드리겠습니다.

---

**사용자 경험 디자인: 다리 놓기**

사용자 경험 디자인(UX 디자인)은 마치 두 개의 섬을 연결하는 다리를 만드는 과정과 같습니다. 한쪽 섬은 사용자가, 다른 쪽 섬은 제품이나 서비스입니다. 사용자들이 두 섬을 쉽게 오갈 수 있도록 다리를 튼튼하고 안전하게 설계해야 합니다. 

1. **사용자 조사: 지도 그리기**
   먼저, 두 섬의 지형을 이해해야 합니다. 이것이 사용자 조사입니다. 사용자의 필요와 불편함을 알아내는 것은 마치 섬의 지도를 그리는 것과 같습니다. 사용자들이 어떤 경로를 원하고, 어떤 장애물이 있는지를 파악하는 것이죠.

2. **정보 구조: 경로 설계**
   다리는 단순

👋 종료합니다.
